In [ ]:
%%bash
# @title 1. Installation de l'environnement (HGP-Reducer, CGAL, dépendances)
set -euo pipefail
WORKDIR="/content"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

echo "Mise à jour et installation de CGAL (Backend sécurisé)..."
apt-get update -qq
apt-get install -y -qq libcgal-dev libgmp-dev libmpfr-dev libboost-all-dev

echo "Installation des bibliothèques Python (UMAP, Scikit-learn, etc.)..."
pip install -q --upgrade pip setuptools wheel Cython cmake pybind11
pip install -q numpy scipy scikit-learn matplotlib umap-learn

echo "Clonage de HGP-clusterer..."
if [ ! -d "HGP-clusterer" ]; then
  # Remplacez l'URL par l'URL de votre dépôt GitHub une fois poussé
  git clone https://github.com/Ludwig-H/HGP-clusterer.git
fi
cd HGP-clusterer
python setup.py build_ext --inplace
echo "Installation terminée ! ✅"

In [ ]:
# @title 1.2 Ajout du chemin vers HGP-Reducer
import sys
import os
if '/content/HGP-clusterer/src' not in sys.path:
    sys.path.append('/content/HGP-clusterer/src')

In [ ]:
# @title 2. Importation et Génération des Benchmarks 3D
import urllib.request
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_swiss_roll, make_s_curve

# --- Dictionnaire pour stocker les jeux de données : "Nom": (Données_3D, Couleurs)
datasets = {}
n_samples = 5000

# ==========================================
# 1. LE MAMMOUTH (Structure Globale)
# ==========================================
url = "https://raw.githubusercontent.com/PAIR-code/understanding-umap/master/raw_data/mammoth_3d.json"
try:
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as response:
        X_mammoth = np.array(json.loads(response.read().decode('utf-8')))
    
    if len(X_mammoth) > 10000:
        np.random.seed(42)
        X_mammoth = X_mammoth[np.random.choice(X_mammoth.shape[0], 10000, replace=False)]
        
    datasets['1. Mammouth'] = (X_mammoth, X_mammoth[:, 1])
except Exception as e:
    print(f"Erreur téléchargement Mammouth : {e}")

# ==========================================
# 2 & 3. SWISS ROLL & S-CURVE (Dépliage)
# ==========================================
X_swiss, c_swiss = make_swiss_roll(n_samples=n_samples, noise=0.05, random_state=42)
datasets['2. Swiss Roll'] = (X_swiss, c_swiss)

X_scurve, c_scurve = make_s_curve(n_samples=n_samples, noise=0.05, random_state=42)
datasets['3. S-Curve'] = (X_scurve, c_scurve)

# ==========================================
# 4. SPHÈRE TRONQUÉE (Topologie)
# ==========================================
u = np.random.uniform(0, 2 * np.pi, n_samples)
v = np.random.uniform(0.4, np.pi - 0.4, n_samples)
x_sph, y_sph, z_sph = np.sin(v) * np.cos(u), np.sin(v) * np.sin(u), np.cos(v)
datasets['4. Sphère Tronquée'] = (np.vstack((x_sph, y_sph, z_sph)).T, z_sph)

# ==========================================
# 5. NOEUD DE TRÈFLE (Démêlage)
# ==========================================
t = np.random.uniform(0, 2 * np.pi, n_samples)
x_knot = np.sin(t) + 2 * np.sin(2 * t)
y_knot = np.cos(t) - 2 * np.cos(2 * t)
z_knot = -np.sin(3 * t)
X_knot = np.vstack((x_knot, y_knot, z_knot)).T + np.random.normal(0, 0.1, (n_samples, 3))
datasets['5. Nœud de Trèfle'] = (X_knot, t)

# ==========================================
# 6. TORE / DONUT (Déchirure)
# ==========================================
u_tor, v_tor = np.random.uniform(0, 2 * np.pi, n_samples), np.random.uniform(0, 2 * np.pi, n_samples)
R, r = 2.0, 1.0
x_tor = (R + r * np.cos(v_tor)) * np.cos(u_tor)
y_tor = (R + r * np.cos(v_tor)) * np.sin(u_tor)
z_tor = r * np.sin(v_tor)
datasets['6. Tore'] = (np.vstack((x_tor, y_tor, z_tor)).T, u_tor)

# --- Affichage
fig = plt.figure(figsize=(18, 10))
fig.suptitle("La Batterie de Crash-Tests 3D (Vérité Terrain)", fontsize=18, fontweight='bold')

for i, (name, (X, color)) in enumerate(sorted(datasets.items())):
    ax = fig.add_subplot(2, 3, i+1, projection='3d')
    ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=color, cmap='Spectral', s=2, alpha=0.8)
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([]); ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# @title 3. Comparaison (t-SNE, UMAP, HGP-Reducer)
from sklearn.manifold import TSNE
import umap
from hgp_reducer import HGPReducer
import time

methods = {
    "t-SNE": TSNE(n_components=2, random_state=42),
    "UMAP": umap.UMAP(n_components=2, random_state=42),
    "HGP-Reducer": HGPReducer(d=2, K=2, laplacian_type='normalized', backend='cgal', verbose=True)
}

n_datasets = len(datasets)
n_methods = len(methods)

fig, axes = plt.subplots(n_methods, n_datasets, figsize=(4 * n_datasets, 4 * n_methods))
fig.suptitle("Comparaison de la Réduction de Dimension", fontsize=24, fontweight='bold')

for col, (name, (X, color)) in enumerate(sorted(datasets.items())):
    axes[0, col].set_title(name, fontsize=16, fontweight='bold')
    
    for row, (method_name, reducer) in enumerate(methods.items()):
        if col == 0:
            axes[row, col].set_ylabel(method_name, fontsize=16, fontweight='bold')
            
        print(f"Exécution de {method_name} sur {name}...")
        start_time = time.time()
        try:
            X_reduced = reducer.fit_transform(X)
            end_time = time.time()
            
            ax = axes[row, col]
            ax.scatter(X_reduced[:, 0], X_reduced[:, 1], c=color, cmap='Spectral', s=2, alpha=0.8)
            ax.set_xticks([]); ax.set_yticks([]); ax.axis('off')
            
            # Ajouter le temps d'exécution
            ax.text(0.05, 0.05, f"{end_time - start_time:.2f}s", transform=ax.transAxes, 
                    fontsize=12, fontweight='bold', bbox=dict(facecolor='white', alpha=0.8))
        except Exception as e:
            print(f"Erreur avec {method_name} sur {name}: {e}")
            axes[row, col].text(0.5, 0.5, "Erreur", ha="center", va="center", color="red")
            axes[row, col].axis('off')

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()